# Тест подключения к Oracle DB из Jupyter Notebook

## Описание
Этот ноутбук демонстрирует минимальное подключение к Oracle Database используя библиотеку `oracledb` в thin mode (без установки Oracle Client).

## Требования
- Python 3.12+
- Библиотека `oracledb` (установка: `pip install oracledb`)
- Доступ к Oracle Database (19c/23c)

## Важные настройки (изменить перед запуском!)
В ячейке **№3** необходимо указать:
1. **USERNAME** — имя пользователя БД (например: `hr`, `scott`, `system`)
2. **DSN** — строка подключения в формате `хост:порт/сервис_имя` (например: `localhost:1521/XEPDB1`)

### Ячейка 1: Импорт необходимых библиотек

In [ ]:
# Импортируем библиотеку oracledb для работы с Oracle Database
import oracledb
# Импортируем модуль getpass для безопасного ввода пароля (скрытый ввод)
import getpass
# Импортируем контекстный менеджер для удобной работы с ресурсами
from contextlib import contextmanager
# Импортируем модуль sys для получения информации о системе и пути к интерпретатору
import sys

### Ячейка 2: Проверка окружения и версий

In [ ]:
# Выводим полный путь к исполняемому файлу Python, чтобы убедиться, что используется правильный интерпретатор (не conda)
print(f"Python: {sys.executable}")
# Выводим версию установленной библиотеки oracledb для совместимости
print(f"oracledb версия: {oracledb.__version__}")
# Сообщаем пользователю, что библиотека успешно найдена в текущем окружении
print("✓ Библиотека найдена в стандартном Python окружении")

### Ячейка 3: НАСТРОЙКА ПОДКЛЮЧЕНИЯ (ИЗМЕНИТЬ ПЕРЕД ЗАПУСКОМ!)

In [ ]:
# === НАСТРОЙКА ===
# Укажите имя пользователя базы данных Oracle (замените "your_username" на реальное имя, например "hr" или "scott")
USERNAME = "your_username"

# Укажите строку подключения в формате: хост:порт/сервис_имя (замените на ваши данные, например "localhost:1521/XEPDB1")
# host - адрес сервера БД (localhost или IP)
# port - порт слушателя Oracle (обычно 1521)
# service_name - имя сервиса базы данных (например, XEPDB1 для Oracle Express Edition)
DSN = "localhost:1521/XEPDB1"

### Ячейка 4: Функция подключения

In [ ]:
# Определяем декоратор контекстного менеджера для безопасного управления подключением к БД
@contextmanager
# Объявляем функцию get_oracle_connection, которая принимает имя пользователя и строку подключения (dsn)
def get_oracle_connection(user: str, dsn: str):
    # Описываем назначение функции в документации (docstring)
    """Контекстный менеджер для безопасного подключения."""
    # Запрашиваем пароль у пользователя в скрытом режиме (символы не отображаются при вводе)
    password = getpass.getpass("Oracle пароль: ")
    # Инициализируем переменную соединения как None перед попыткой подключения
    conn = None
    # Начинаем блок обработки исключений для безопасной работы с БД
    try:
        # Устанавливаем соединение с Oracle, передавая пользователя, пароль и строку подключения (dsn)
        conn = oracledb.connect(user=user, password=password, dsn=dsn)
        # Выводим сообщение об успешном подключении
        print("✓ Подключение успешно")
        # Возвращаем объект соединения в контекст (yield) для использования в блоке with
        yield conn
    # Ловим специфичные ошибки библиотеки oracledb, если подключение или запросы не удались
    except oracledb.Error as e:
        # Выводим сообщение об ошибке подключения с деталями исключения
        print(f"✗ Ошибка подключения: {e}")
        # Пробрасываем исключение дальше, чтобы выполнение кода остановилось или было обработано выше
        raise
    # Блок finally выполняется всегда, независимо от наличия ошибок, для очистки ресурсов
    finally:
        # Проверяем, существует ли активное соединение (не равно None)
        if conn:
            # Закрываем активное соединение с базой данных для освобождения ресурсов
            conn.close()
            # Выводим сообщение об успешном закрытии соединения
            print("✓ Соединение закрыто")

### Ячейка 5: Запуск тестирования

In [ ]:
# === ТЕСТИРОВАНИЕ ===
# Выводим заголовок начала тестирования подключения в консоль вывода ноутбука
print("\n=== ТЕСТ ПОДКЛЮЧЕНИЯ ===")
# Используем контекстный менеджер для автоматического открытия и закрытия соединения (пароль будет запрошен при выполнении)
with get_oracle_connection(USERNAME, DSN) as conn:
    # Создаем объект курсора из активного соединения для выполнения SQL-запросов
    cursor = conn.cursor()
    
    # 1. Базовый тест DUAL
    # Выполняем простейший SQL-запрос к системной таблице dual для проверки работоспособности соединения
    cursor.execute("SELECT 'OK' as status FROM dual")
    # Получаем первую строку результата запроса (кортеж) и выводим первый элемент как статус теста
    print("✓ DUAL тест:", cursor.fetchone()[0])
    
    # 2. Информация о БД
    # Выполняем запрос к системному представлению v$version для получения версии СУБД Oracle (ограничиваем одной строкой через rownum)
    cursor.execute("SELECT banner FROM v$version WHERE rownum = 1")
    # Получаем результат запроса (полная строка баннера), берем первые 60 символов и добавляем многоточие для краткости вывода
    print("✓ Oracle версия:", cursor.fetchone()[0][:60] + "...")
    
    # 3. Счетчик объектов (проверка доступа)
    # Выполняем подсчет количества объектов (таблицы, индексы, процедуры и т.д.) в текущей схеме пользователя
    cursor.execute("SELECT COUNT(*) as objects FROM user_objects")
    # Получаем числовое значение количества объектов и выводим его в консоль с пояснением
    print("✓ Объектов в схеме:", cursor.fetchone()[0])

### Ячейка 6: Завершение

In [ ]:
# Выводим финальное сообщение об успешном завершении теста и готовности кода к использованию с эмодзи для наглядности
print("\n🎉 ГОТОВО! Код протестирован и готов к публикации")
# Выводим напоминание пользователю о необходимости замены параметров USERNAME и DSN на актуальные значения перед реальным использованием
print("📋 Для запуска: замените USERNAME и DSN на свои параметры")